# Bring your own data — minimal end-to-end example

This notebook builds the same dataset shape as the paper, but from a tiny synthetic input set written to `examples/_byo_demo/`. **No network, no TEASER, no TABULA** — useful as a template when adapting the pipeline to a different country or data source.

What it does, cell by cell:
1. Write 2 synthetic archetypes to `_byo_demo/input/archetypes.parquet`.
2. Write 2 synthetic electricity profiles to `_byo_demo/input/E.parquet`.
3. Write a 1-row location mapping CSV.
4. Write a YAML config that points the four providers at those files.
5. Run the pipeline (B / E / O steps; weather + simulation skipped).

To adapt the pipeline to your own data, replace the `_make_*` cells with your real files (keeping the canonical schema column names) and point the YAML providers at them.

The matching script is `examples/02_byo_country.py` — same content, different format.

In [ ]:
import sys, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.pipeline import run

DEMO = ROOT / 'examples' / '_byo_demo'
if DEMO.exists():
    shutil.rmtree(DEMO)
DEMO.mkdir(parents=True)
print(f'Demo root: {DEMO}')

## 1. Synthetic archetypes (B.parquet)

Two SFH archetypes with hand-curated 1R1C parameters. The columns are exactly `ArchetypeSchema.REQUIRED_1R1C` — same shape the pipeline expects.

In [ ]:
archetypes = pd.DataFrame({
    'archetype_id':       [1, 2],
    'construction_year':  [1980, 2015],
    'area_m2':            [120.0, 150.0],
    'n_floors':           [2, 2],
    'height_floor_m':     [2.5, 2.6],
    'thermal_resistance': [0.0030, 0.0080],
    'thermal_capacitance':[3.0e7, 5.0e7],
})
(DEMO/'input').mkdir(parents=True, exist_ok=True)
archetypes.to_parquet(DEMO/'input/archetypes.parquet', index=False)
archetypes

## 2. Synthetic electricity (E.parquet)

Two synthetic profiles for one full year at 15-min resolution: a workday-shaped profile (evening peak) and a flat profile.

In [ ]:
year = 2024
idx = pd.date_range(f'{year}-01-01', f'{year}-12-31 23:00', freq='15min', tz='UTC')
rng = np.random.default_rng(0)
t = idx.hour + idx.minute / 60.0

base_a = 200 + 600 * np.exp(-((t - 19) ** 2) / 4)   # evening peak
base_b = np.full_like(base_a, 350.0)                # flat
noise  = rng.normal(0, 30, len(idx))

elec = pd.concat([
    pd.DataFrame({'timestamp': idx, 'profile_id': 1, 'electricity_demand': base_a + noise}),
    pd.DataFrame({'timestamp': idx, 'profile_id': 2, 'electricity_demand': base_b + noise}),
], ignore_index=True)
elec.to_parquet(DEMO/'input/E.parquet', index=False)
print(f'{len(elec):,} rows; first 3:')
elec.head(3)

## 3. Locations and YAML config

One dummy location and a YAML that wires all four providers to local files.

In [ ]:
(DEMO/'output').mkdir(parents=True, exist_ok=True)
pd.DataFrame({'location_id':[1], 'latitude':[50.0], 'longitude':[10.0]}).to_csv(
    DEMO/'output/location_mapping.csv', index=False)

cfg = {
    'year': year,
    'output_dir': str(DEMO/'output'),
    'thermal_model': 'r1c1',
    'locations': 'all',
    'archetype_provider':   {'type': 'csv',     'path': str(DEMO/'input/archetypes.parquet')},
    'electricity_provider': {'type': 'parquet', 'path': str(DEMO/'input/E.parquet')},
    'occupancy_provider':   {'type': 'geoma',   'alpha': 0.05},
    'weather_provider':     {'type': 'csv',
                             'weather_dir':       str(DEMO/'output/weather'),
                             'location_mapping':  str(DEMO/'output/location_mapping.csv')},
    'simulation': {'heating_setpoint_C': 20.0, 'cooling_setpoint_C': 26.0,
                   'inhabitants': 2, 'gains_per_person_W': 65,
                   'resolution': '60min', 'overwrite': False},
}
with open(DEMO/'config.yml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(DEMO/'config.yml').read())

## 4. Run the pipeline

Weather and simulation skipped — the demo only exercises the input parquets (B, E, O) and the location mapping.

In [ ]:
result = run(
    DEMO/'config.yml',
    overwrite=False,
    skip_weather_fetch=True,
    skip_simulation=True,
)
print(result)

In [ ]:
# Verify outputs
print('Files in', DEMO/'output:')
for p in sorted((DEMO/'output').rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(DEMO)}  ({p.stat().st_size:,} bytes)')